## Evaluating predictions on testing data

In [ ]:
import sys
import importlib as imp
import math
import numpy as np
import torch
import rasterio
import matplotlib.pyplot as plt
from rasterio.windows import Window
from sklearn.metrics import mean_squared_error, mean_absolute_error

from utils import utils
from visualizer import plots
from analyzer import analyze_predictions
from data_builder import data_methods

In [ ]:
print(f"python version = {sys.version}")
print(f"numpy version = {np.__version__}")
print(f"pytorch version = {torch.__version__}")

In [ ]:
# GET config
EXP_NAME = "exp319"

config = utils.get_config(EXP_NAME)
config["mode"] = "inference"
directory_paths = utils.get_directories(config["machine"])

In [ ]:
nsamples_to_plot = 100_000

for year in (2020,):
    print(" --- " + str(year) + "---")
    config["inference_years"] = (year,)

    # TILE THE PREDICTIONS TOGETHER
    model_name = utils.get_model_name(config["expname"], config["seed"])
    mosaic_filename = (
        directory_paths["mosaics_dir"]
        + model_name
        + "_"
        + str(config["inference_years"][0])
        + "_mlhfi_mosaic.tif"
    )
    labels_filename = (
        directory_paths["data_dir"]
        + "hii_"
        + str(config["inference_years"][0])
        + "-01-01_uint8.tif"
    )

    with rasterio.open(mosaic_filename) as predict_tif:

        if isinstance(config["data"]["inference_region"], dict):
            lat_s, lat_n, lon_w, lon_e = data_methods.get_region_bounds(
                region=config["data"]["inference_region"],
                tile_len_deg=config["tile_len_deg"],
            )
        else:
            lat_s, lat_n, lon_w, lon_e = config["data"]["inference_region"]
        print(lat_s, lat_n, lon_w, lon_e)

        ilat_n, ilon_w = predict_tif.index(lon_w, lat_n, op=np.ceil)
        ilat_s, ilon_e = predict_tif.index(lon_e, lat_s, op=np.ceil)
        ilat_n, ilon_w = max(ilat_n, 0), max(ilon_w, 0)

        window = Window.from_slices((ilat_n, ilat_s + 1), (ilon_w, ilon_e + 1))
        predictions = np.asarray(predict_tif.read(1, window=window), dtype="float")

        with rasterio.open(labels_filename) as label_tif:
            window = label_tif.window(*predict_tif.bounds)
            labels = np.asarray(label_tif.read(1, window=window, boundless=True), dtype="float")

    # ALIGN and PROCESS PREDICTIONS FOR ANALYAIS
    labels, predictions = analyze_predictions.process_flatten(
        config, labels, predictions, remove_edges=True
    )

    # %%
    # MAKE HISTOGRAMS
    plt.figure(figsize=(20, 4.5))
    plt.subplot(1, 3, 1)
    bins = np.arange(-0.5, 100.5, 1)
    plt.hist(labels, bins, density=True)
    plt.title(config["expname"] + ": labels")
    plt.ylim(0, 0.05)

    plt.subplot(1, 3, 2)
    bins = np.arange(-0.5, 100.5, 1)
    plt.hist(predictions, bins, density=True)
    plt.title(config["expname"] + ": predictions")
    plt.ylim(0, 0.05)

    # MAKE SUMMARY FIGURE
    rmse = np.sqrt(mean_squared_error(labels, predictions)).round(4)
    mae = mean_absolute_error(labels, predictions).round(4)
    corr = np.corrcoef(labels, predictions)[0, 1].round(4)

    rng = np.random.default_rng(42)
    iplot = rng.choice(np.arange(len(predictions)), size=nsamples_to_plot)

    plt.subplot(1, 3, 3)
    inc = 2
    plt.hist2d(
        x=labels[iplot],
        y=predictions[iplot],
        density=False,
        norm="log",
        bins=np.arange(0, 100 + inc, inc),
        cmap="Spectral_r",
        vmin=1,
    )
    plt.colorbar()
    plt.plot((0, 100), (0, 100), "-", linewidth=2, color="k", alpha=0.75)
    plt.title(f"{rmse = }, {mae = }, {corr = }")
    plt.xlabel("labels")
    plt.ylabel("predictions")
    plt.xlim(0, 100)
    plt.ylim(0, 100)

    plots.savefig(config, str(year) + "_prediction_metrics")
    plt.show()

## Save predictions with flagged differences from the HII labels

In [ ]:
# raise ValueError

In [ ]:
with rasterio.open(mosaic_filename) as predict_tif, rasterio.open(labels_filename) as label_tif:
    data1 = predict_tif.read(1)

    # extract metadata from the first raster
    meta = predict_tif.meta.copy()
    meta["compress"] = "lzw"
    meta["count"] = 2

    # read the window of the second raster with the same extent as the first raster
    window = label_tif.window(*predict_tif.bounds)

    # read the data from the second raster with the same window as first raster
    data2 = label_tif.read(1, window=window, boundless=True)
    data2 = np.where(data1 == predict_tif.nodata, predict_tif.nodata, data2)

    # calculate the difference
    data = data1 - data2
    data[np.abs(data)<30] = 200
    data[data<0] = 0
    data[(data>0) & (data<=101)] = 100
    data[data==200] = 50
    data = np.where(data2 == label_tif.nodata, predict_tif.nodata, data)

    data = np.asarray(np.round(data), dtype="uint8")

    new_data = np.zeros((2, data1.shape[0], data1.shape[1]), dtype="uint8")
    new_data[0,:,:] = data1
    new_data[1,:,:] = data

    # write the result to a new raster
    with rasterio.open(mosaic_filename[:-4] + "_flaggedDiffs.tif", 'w', **meta) as dst:
        dst.write(new_data)
        dst.set_band_description(1, "prediction")
        dst.set_band_description(2, "flagged_diffs")